In [1]:
import pandas as pd
import numpy as np
import requests

DATA_PATH = '../data'

full_dataset = pd.read_parquet(f'{DATA_PATH}/full_dataset.parquet')
movies = pd.read_csv(f'{DATA_PATH}/movies.csv')

print("Loaded successfully")
print(full_dataset.shape)

Loaded successfully
(50000190, 34)


In [2]:
import requests

# Get weather for Rochester, NY ( my location )
url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 43.1566,
    "longitude": -77.6088,
    "current_weather": True
}
response = requests.get(url, params=params)
print(response.json())

{'latitude': 43.15501, 'longitude': -77.596855, 'generationtime_ms': 0.14710426330566406, 'utc_offset_seconds': 0, 'timezone': 'GMT', 'timezone_abbreviation': 'GMT', 'elevation': 161.0, 'current_weather_units': {'time': 'iso8601', 'interval': 'seconds', 'temperature': '°C', 'windspeed': 'km/h', 'winddirection': '°', 'is_day': '', 'weathercode': 'wmo code'}, 'current_weather': {'time': '2026-03-30T01:45', 'interval': 900, 'temperature': 7.6, 'windspeed': 5.7, 'winddirection': 108, 'is_day': 0, 'weathercode': 3}}


In [3]:
def get_weather_condition(weathercode, temperature):
    if weathercode in [51, 53, 55, 61, 63, 65, 80, 81, 82, 95, 96, 99]:
        return 'rainy'
    elif weathercode in [71, 73, 75, 77, 85, 86]:
        return 'snowy'
    elif weathercode in [0, 1] and temperature >= 28:
        return 'sunny'
    else:
        return None  # normal day — no weather row

condition = get_weather_condition(3, 7.5)
print(f"Condition: {condition}")

Condition: None


In [ ]:
# Map weather condition to boosted genres
weather_genre_map = {
    'rainy': ['Drama', 'Romance', 'Thriller', 'Mystery', 'Film-Noir'],
    'snowy': ['Animation', 'Children', 'Fantasy', 'Musical', 'Comedy'],
    'sunny': ['Action', 'Adventure', 'Comedy']
}

def get_weather_recommendations(condition, full_dataset, movies, n=10):
    if condition is None:
        return None

    boosted_genres = weather_genre_map[condition]

    candidates = full_dataset.drop_duplicates('movieId').copy()
    candidates = candidates[candidates['movie_rating_count'] >= 500]
    candidates = candidates[candidates['movie_avg_rating'] >= 3.5]

    genre_mask = candidates[boosted_genres].sum(axis=1) > 0
    candidates = candidates[genre_mask]

    sample = candidates.sample(n=n, random_state=None)

    # Keep avg rating before merge
    sample = sample[['movieId', 'movie_avg_rating']].copy()
    sample['movieId'] = sample['movieId'].astype(str)
    movies['movieId'] = movies['movieId'].astype(str)
    result = sample.merge(movies, on='movieId')

    return result[['title', 'genres', 'movie_avg_rating']]

In [ ]:
print("RAINY:")
print(get_weather_recommendations('rainy', full_dataset, movies))
print("\nSNOWY:")
print(get_weather_recommendations('snowy', full_dataset, movies))
print("\nSUNNY:")
print(get_weather_recommendations('sunny', full_dataset, movies))